#### Youtube Video Chat RAG Bases System

In [61]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4.1-nano", model_provider="openai")

In [62]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY")

from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [63]:
## Function that extract the youtube Video id from function
import re

def extract_youtube_id(url: str) -> str | None:
    """Extract YouTube video ID from a URL."""
    match = re.search(r"(?:v=|\/)([0-9A-Za-z_-]{11})", url)
    return match.group(1) if match else None

In [98]:
video_id = extract_youtube_id("https://www.youtube.com/watch?v=MMS04bku3FE&t=799s")

In [99]:
video_id

'MMS04bku3FE'

#### Step-1 Data Ingestion(Data/Document loading)

In [102]:
language = ['en']
language

['en']

In [103]:
from youtube_transcript_api import YouTubeTranscriptApi
youtube_video_id = video_id
transcript_list = YouTubeTranscriptApi().fetch(youtube_video_id, languages=language)


In [104]:
video_transcript = " ".join(snippet.text for snippet in transcript_list)

In [105]:
video_transcript

"Hello guys. So we are going to continue a discussion with respect to rag. Already in our previous video if you remember we have completed the entire pipeline from data injection to chunking to embedding and finally converting the text to vectors and storing into a vector store which was stored locally. Now in this particular video I want to show you an example wherein after we convert the text into vectors we store this into a vector DB that is hosted in a cloud. Okay. So for this we are going to use this amazing platform which is called as Typesense and thank you Typesense for sponsoring this video. Now for all those people who do not know about Typesense. It is lightning fast open-source search and here it is designed for use cases such as website mobile search. It provides amazing features like instance search uh natural language support semantic and vector search capabilities. Right? So if you just go ahead and search for something right let's say that there are so many different 

In [106]:
from langdetect import detect
from deep_translator import GoogleTranslator

def translate_to_english(text, max_chunk_len=500):
    detected_lang = detect(text)
    if detected_lang == "en":
        return text  

    chunks = [text[i:i+max_chunk_len] for i in range(0, len(text), max_chunk_len)]
    translated_chunks = [
        GoogleTranslator(source=detected_lang, target="en").translate(chunk)
        for chunk in chunks
    ]
    return " ".join(translated_chunks)


In [107]:
result = translate_to_english(video_transcript)
print(result)

Hello guys. So we are going to continue a discussion with respect to rag. Already in our previous video if you remember we have completed the entire pipeline from data injection to chunking to embedding and finally converting the text to vectors and storing into a vector store which was stored locally. Now in this particular video I want to show you an example wherein after we convert the text into vectors we store this into a vector DB that is hosted in a cloud. Okay. So for this we are going to use this amazing platform which is called as Typesense and thank you Typesense for sponsoring this video. Now for all those people who do not know about Typesense. It is lightning fast open-source search and here it is designed for use cases such as website mobile search. It provides amazing features like instance search uh natural language support semantic and vector search capabilities. Right? So if you just go ahead and search for something right let's say that there are so many different k

In [108]:
# Save transcript to a text file
with open("youtube_transcript.txt", "w", encoding="utf-8") as f:
    f.write(result)

print("Transcript saved successfully as youtube_transcript.txt")


Transcript saved successfully as youtube_transcript.txt


In [111]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("youtube_transcript.txt", encoding="utf-8")
documents = loader.load()

##### Text splitter

In [112]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

In [113]:
len(chunks)

52

In [114]:
chunks[0].page_content

'Hello guys. So we are going to continue a discussion with respect to rag. Already in our previous video if you remember we have completed the entire pipeline from data injection to chunking to embedding and finally converting the text to vectors and storing into a vector store which was stored locally. Now in this particular video I want to show you an example wherein after we convert the text into vectors we store this into a vector DB that is hosted in a cloud. Okay. So for this we are going'

#### Embedding Generation and storing


In [115]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)

In [35]:
vectorstore.index_to_docstore_id

{0: '0a00bfe5-a9f5-4796-8537-d64f5fc89f63',
 1: '9a79db28-2ba3-483f-b83d-d5090fc8f0c3',
 2: '0fe129c9-213e-44b7-8ffb-691f9c1138ee',
 3: '2b4bf14a-75e5-4a71-9ff9-0ec3b270fa99',
 4: 'e7350620-dc1c-4533-9c2c-a6aacb335da1',
 5: 'bab75d0d-690f-40b3-8803-7b6aa325d44a',
 6: '4d284973-e4ea-40ce-9983-05a0e614a64e',
 7: '1cd44b83-31ee-4742-8c42-635646218ab4',
 8: '70445afd-0ff0-418d-a2ed-a748f234f8ec',
 9: '83d3a00e-1b59-4d3c-9e2b-bc865a1b2e58',
 10: '45a5a5ec-211b-486a-ada1-cde232800e16',
 11: 'fdc4059d-b23b-4a19-b8bf-7cc2bdaa4f43',
 12: '4232e7d5-e3c5-4cbd-9faa-56017c19f55c',
 13: 'aa83989e-3e64-4492-b9c6-6470168ce607',
 14: '95172e27-7163-43b9-a5a7-6b8c4a670bbc',
 15: '852eeeb8-4a6e-422d-90c3-7db2ba30906c',
 16: 'ea223963-fb2e-472b-b2d3-2151be51d905',
 17: '7f47973e-0e95-40fa-a0c6-6649f9bbcdb4',
 18: 'bfd8791c-8598-42be-a307-5626c5cb2e2d',
 19: 'd13cc668-1ff2-4190-a6ec-da68096a8703',
 20: 'b71c2b9f-2101-4612-8a8e-62b862f7c1a7',
 21: '6fba384c-2953-413b-b8fd-86ae5b66f905',
 22: '7a828322-1a2e-

In [116]:
vectorstore.get_by_ids(['2b6c27f6-3eca-4854-998f-9f0681d3c1b3'])

[]

#### Step 2 Retrieval

In [117]:
retrieval = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})



In [118]:
retrieval

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000025303B35B40>, search_kwargs={'k': 2})

In [119]:
retrieval.invoke('What is rag')

[Document(id='7f83a5e3-f3c5-4513-a8ef-7ee82549174f', metadata={'source': 'youtube_transcript.txt'}, page_content='Hello guys. So we are going to continue a discussion with respect to rag. Already in our previous video if you remember we have completed the entire pipeline from data injection to chunking to embedding and finally converting the text to vectors and storing into a vector store which was stored locally. Now in this particular video I want to show you an example wherein after we convert the text into vectors we store this into a vector DB that is hosted in a cloud. Okay. So for this we are going'),
 Document(id='ad51e074-d6e0-49eb-80a0-a709bc7d7e81', metadata={'source': 'youtube_transcript.txt'}, page_content="this ipv I will go ahead and select my kernel and then we will start writing the code over here. And the code this will be with respect to creating a rag application. Rag application using type sense. Okay. Using type sense. Perfect. Now once this is done first of all y

#### Step 3 Generation

In [40]:
from langchain_core.prompts import PromptTemplate

template = """
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

Context: {context}
Question: {question}
"""

#
prompt = PromptTemplate.from_template(template)

In [ ]:
from langchain.schema.runnable import RunnablePassthrough, RunnableParallel
from langchain.schema.output_parser import StrOutputParser

In [42]:
str_output_parser = StrOutputParser()

In [56]:
parallel_chain = RunnableParallel({
    "context": retrieval,
    "question": RunnablePassthrough()
}
)

In [57]:
parallel_chain.invoke("What is multimodal rag system?")

{'context': [Document(id='0fe129c9-213e-44b7-8ffb-691f9c1138ee', metadata={'source': 'youtube_transcript.txt'}, page_content="now in this video we'll discuss the complete and detailed introduction of this multimodel rag now here this uh uh discussion I have divided into multiple points what is a rag rag architecture component of the rag architecture then from the fourth Point itself my main thing is going to be start so from here multimodel rag what is a multimodel rag multimodel embedding and multimodel generation so this is the hard or Cod part of this multimodel guys multimodel R system so please don't miss out"),
  Document(id='45a5a5ec-211b-486a-ada1-cde232800e16', metadata={'source': 'youtube_transcript.txt'}, page_content='I think you click L on this video so the main point is what is a multimodel rag so here you can read the definition carefully so multimodel rack is a AI system that can understand different modelist like images audio video text so in the classical Rec system w

In [58]:
rag_chain = (
    parallel_chain|
    prompt |
    llm |
    str_output_parser
)

In [59]:
rag_chain.invoke("what is multimodal rag system?")

'A multimodal RAG system is an AI system that can understand different modalities such as images, audio, video, and text.'